# Lab 06 · Image and NLP Enrichment (Notebook Walkthrough)

**Concept.** The `visual_nlp` profile adds `OcrSkill`, `ImageAnalysisSkill`, and `LanguageDetectionSkill` to the Blob skillset, and switches the Search-managed extractor to the **Document Layout** skill (`AZURE_SEARCH_SKILLSET_PREFERRED_EXTRACTOR=document_layout`). This matters for a diagram-heavy engineering document where evidence can live in figures, not just prose.

**Two ways to crop a figure (complementary, not duplicate):**
- **Document Layout skill (server-side).** Runs inside the skillset on the billable Foundry resource. With `extractionOptions=['images','locationMetadata']` it detects figure regions, crops them, and emits each crop under `/document/normalized_images/*` together with its `pageNumber` + `boundingPolygons`. The skillset's **knowledge-store projections** persist the crops to `search-image-assets`, the location metadata to `search-image-assets-meta`, and (for the `visual_nlp` profile) each figure's OCR + caption text to `search-image-assets-text` so chart content can travel with the chunk.
- **Offline parser (local).** Renders each PDF page and crops figure regions, attaching thumbnails to chunks for the citation UI. It is the fallback when the native/server-side path is off.

**Why the extractor swap changes OCR quality.** With the default **Document Extraction** extractor the indexer's built-in cracker emits *whole-page* normalized images, so `OcrSkill` and `ImageAnalysisSkill` (both at `/document/normalized_images/*`) run on entire pages and mostly capture page noise. With **Document Layout** those same skills run on *figure-scoped crops*, so OCR reads the figure's own labels. On the deep-excavation PDF this took non-empty OCR results from ~12 (whole-page) to ~63 (figure-aware).

**Where each signal lands:** the per-image caption (`captions/*/text`) and OCR collections populate the **Search-managed enrichment index** (`...-visual-nlp`) that **Agentic** retrieval can draw on; the app also merges a **document-level `image_description_text`** back onto every **canonical chunk**, so `full_text` / `vector` / `hybrid` gain a visual signal too.

**Key lesson:** caption *quality* is document-dependent. Image Analysis 3.2 returns descriptive but generic captions (“chart”, “diagram”, “a diagram of a house”); the big win here is OCR density + figure scoping + page/polygon **location metadata**. Always **verify what actually landed**.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': '<home>\\rag-on-azure-lab',
 'env_values_loaded': 0,
 'search_endpoint': 'https://your-search-service.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Confirm the active extractor

The `visual_nlp` profile uses the Document Layout extractor when `AZURE_SEARCH_SKILLSET_PREFERRED_EXTRACTOR=document_layout` and a billable Foundry resource is attached. When Document Layout is active the indexer runs with `imageAction: none` (the skill does its own figure-aware cropping, so the built-in whole-page cracker is turned off).

In [2]:
from backend.core.config import settings
print('preferred extractor      :', settings.azure_search_skillset_preferred_extractor)
print('document layout available:', settings.azure_search_document_layout_skill_available)
print('image serving enabled    :', settings.azure_search_enable_image_serving)

preferred extractor      : document_layout
document layout available: True
image serving enabled    : True


## Step 3 — Ingest with image + NLP enrichment

In [3]:
job = lab.ingest(skill_profile='visual_nlp', reuse=True)
lab.chunk_overview(job)

Reusing existing 'visual_nlp' ingestion (doc_id=68c609c0, 412 chunks). Pass reuse=False to force a fresh run.


{'doc_id': '68c609c0e1ca42309afa4521e1e888e9',
 'skill_profile': 'visual_nlp',
 'chunk_count': 412,
 'avg_tokens': 200.4,
 'max_tokens': 777,
 'chunks_with_summary': 412,
 'chunks_with_keyword_hints': 412,
 'chunks_with_image_description': 412}

`chunks_with_image_description` is now **non-zero** - every chunk carries the merged document-level `image_description_text` assembled from the figure-scoped Image Analysis captions. The summaries and keyword hints from Lab 05 are still present too.

## Step 4 — Inspect the server-side enrichment (OCR + captions)

The Document Layout + OCR + Image Analysis skills wrote a per-image caption and OCR collection into the `...-visual-nlp` enrichment index. Because the skills ran on **figure crops** (not whole pages), the OCR is dense and figure-scoped - real diagram labels rather than page noise.

In [4]:
vis = lab.enrichment_visual_fields(job)
print('enrichment index   :', vis['index_name'])
print('detected language  :', vis.get('detected_language'))
print('caption count      :', vis.get('image_description_count'))
print('caption sample     :', vis.get('image_description_sample'))
print('OCR results        :', vis.get('ocr_count'),
      '| non-empty:', vis.get('ocr_nonempty_count'))
for t in vis.get('ocr_sample', []):
    print('   OCR:', t[:120])

enrichment index   : ai-search-lab-enrichment-index-visual-nlp
detected language  : en
caption count      : 71
caption sample     : ['a collage of a construction site', 'chart', 'diagram', 'diagram', 'a diagram of a house']
OCR results        : 71 | non-empty: 63
   OCR: FRE VAR ..
   OCR: GIF03 GIF03 2 (Piezometer) (Standpipe) 1.0 mPD 1 GIF03 (Standpipe) Fill -3.5mPD Excavated 0 side Marine -1 Clay Piezomet
   OCR: Fillet weld Channel planking Waling beam Fillet weld
   OCR: Interlock at centre line of the wall Type U sheet pile Short stud Waling
   OCR: Interlock at outer face of wall section Type Z sheet pile Waling


## Step 5 — Inspect the figure crops + location metadata

The skillset's **knowledge-store projections** persisted each figure crop to blob, plus a metadata object carrying the figure's `pageNumber` and `boundingPolygons`. This is the Azure-native, *location-aware* equivalent of the offline parser's local figure PNGs - and it is what `/api/native-images` serves back as citation evidence.

In [5]:
import pandas as pd

assets = lab.figure_assets(job, limit=5)
print('figure crops in search-image-assets      :', assets['crops'])
print('metadata objects in search-image-assets-meta:', assets['metadata'])
pd.DataFrame(assets['samples'])

figure crops in search-image-assets      : 68
metadata objects in search-image-assets-meta: 71


,image,page,bounding_polygons
0,1_2525ea1f-ceb9-4929-8841-f1e5ce4ef8ff.jpg,1,"[[{""x"":0.5272,""y"":2.4663},{""x"":7.7308,""y"":2.46..."
1,18_e3c66d8f-098e-4eae-9f7b-381cf8554dc2.jpg,18,"[[{""x"":0.8878,""y"":6.2818},{""x"":7.5535,""y"":6.26..."
2,33_2e34621d-81ad-4a55-96a6-7e1f395a60cc.jpg,33,"[[{""x"":1.2833,""y"":1.2283},{""x"":6.9076,""y"":1.22..."
3,35_01c8e3a6-e2e0-441b-bd31-4184fd3831fd.jpg,35,"[[{""x"":1.3616,""y"":1.5444},{""x"":6.8654,""y"":1.54..."
4,35_a7bb64a9-218b-4f1e-805e-7daa3fb2701e.jpg,35,"[[{""x"":1.3843,""y"":5.4691},{""x"":6.9232,""y"":5.46..."


> Note: with the native/server-side path active, figures come from the **knowledge store**, not the offline parser, so canonical chunks here carry no `image_evidence` thumbnails - the figure evidence lives in `search-image-assets` instead. Set `ENABLE_PARSER_FIGURE_EXTRACTION=true` (and disable native multimodal) if you want to compare the parser's page-rendered thumbnails side by side.

### Making figure *text* travel with the chunk (chart-aware answers)

Serving the crop image is great for a human reader, but in the direct lanes (`Full text`, `Vector`, `Hybrid`) the **answer model only sees the citation's text snippet** - it can't read pixels. So a chart whose key numbers live inside the image would not be reflected in the answer even when the figure is cited.

To close that gap, the skillset adds a third knowledge-store projection - an **inline-shaped object projection** - that captures each figure's `ocrText` (chart numbers, axis labels) and `caption`, keyed by `pageNumber`, into a dedicated `search-image-assets-text` container. The base object projection above only serializes the image node (path + page), *not* the enriched `ocr_text_chunks` / `image_analysis` siblings, which is why a separate, explicitly-shaped projection is required. At query time `_hydrate_citations` joins that text onto the citation snippet by page, so the figure's content flows into both the deterministic direct-search answer and the app-side LLM prompt - every retrieval mode becomes chart-aware.

> **Takes effect on the next ingestion.** The text projection only writes blobs when the indexer next runs against the `visual_nlp` profile; until then the query-side join is a safe no-op (empty container), so nothing regresses on the already-indexed corpus. We are **not** re-indexing in this lab - the chart text lights up automatically the next time documents are ingested.


## Step 6 — Hybrid vs Agentic on a visual question

**Hybrid** now benefits directly: the merged `image_description_text` is part of every canonical chunk, so visual vocabulary (“construction site”, “chart”) is searchable. **Agentic** can additionally consult the Search-managed enrichment index, where the *full* per-image caption and (now much richer) OCR collections live, and tends to assemble more cross-section citations.

In [6]:
QV = 'What do the figures on ground settlement and wall deflection show, and what visual evidence supports the answer?'

print('--- HYBRID (canonical chunk index, now carries merged captions) ---')
resp_hybrid = lab.ask(QV, job=job, retrieval_mode='hybrid', record_as='lab06_visual_hybrid')
lab.show_answer(resp_hybrid, max_citations=4)

--- HYBRID (canonical chunk index, now carries merged captions) ---


[retrieval_mode=hybrid | scoring_profile=default | citations=7]

However, this method only calculates the lateral deflection of the embedded wall and estimation of ground settlement is usually based on empirical correlations. Lui & Yau (1995) reported the back analysis of the basement excavation at the Dragon Centre, Kowloon. [1]

Supporting evidence:
- 8v / He (%) ôv = 0.50 8h 8h / Hẹ (%) where 8h = Maximum lateral wall deflection He = Maximum excavation depth Figure 8.4 Relationship between Ground Settlement and Lateral Wall Deflection from Case Study Data Consolidation settlements are seldom estimated using empirical methods. Instead, they are commonly assessed by carrying out one-dimensional consolidati [6]
- Clough & O'Rourke (1990) discussed typical profiles of wall deflection and the adjacent ground deformation based on case histories (Figure 8.3). During the initial stage, dewatering and soil excavation are carried out before installation of the first lateral support. [3]

Cita

In [7]:
print('--- AGENTIC (also reads the enrichment index: full captions + OCR) ---')
resp_agentic = lab.ask(QV, job=job, retrieval_mode='agentic', record_as='lab06_visual_agentic')
lab.show_answer(resp_agentic, max_citations=6)

--- AGENTIC (also reads the enrichment index: full captions + OCR) ---


[retrieval_mode=agentic | scoring_profile=default | citations=8]

- The figures show that **ground settlement and wall deflection are related, but not always strongly correlated**. For small deformations, the publication says **there is no apparent correlation** between maximum ground settlement and lateral wall deflection, while for larger deformations the **ground settlement is usually about 50% of the maximum wall deflection**.[1][2]
- The figures also show that in some case histories the settlement can be **about 0.75 to 1.0 times the wall deflection**, indicating that the relationship varies by project and ground conditions.[3]
- Visual evidence supporting this is the publication’s **Figure 8.4, “Relationship between Ground Settlement and Lateral Wall Deflection from Case Study Data,”** which plots settlement against wall deflection and shows the ratio ranges and scatter of the monitored projects.[2][4]
- Additional visual/contextual support comes from **Figure 8.3, “Typical Profi

## Takeaways

- The **Document Layout** extractor crops figures *server-side* and records each figure's `pageNumber` + `boundingPolygons` to the knowledge store - figure-aware **and** location-aware.
- Because OCR and Image Analysis now run on figure crops (not whole pages), the OCR is dramatically richer (real diagram labels, not page noise). Captions stay generic - judge the *quality* of each signal, not just its presence.
- The server-side knowledge-store crops and the offline parser thumbnails solve the same problem in complementary ways; with the native path on, figures come from the knowledge store and are served via `/api/native-images`.
- A dedicated `search-image-assets-text` projection captures each figure's OCR + caption text and `_hydrate_citations` splices it onto citations by page, so the figure's chart content travels with the chunk and **every** retrieval mode can answer about chart numbers (effective on the next ingestion).
- Hybrid gains the merged visual summary on every chunk; Agentic additionally taps the full caption/OCR collections in the enrichment index.

Next: **Lab 07** focuses on agentic retrieval mechanics (planning, subqueries, grounded synthesis).
